# Real Estate Investment Prediction - Machine Learning Models

This notebook trains and evaluates models for two tasks:
- **Regression**: Predict the future price of the house after 5 years.
- **Classification**: Predict whether the property is a good investment.

Models used:
- Regression: Linear Regression, Random Forest Regressor, XGBoost Regressor
- Classification: Logistic Regression, Random Forest Classifier, XGBoost Classifier

Metrics:
- Regression: RMSE, MAE, R² Score
- Classification: Accuracy, Precision, Recall, F1-score, Confusion Matrix

In [1]:
pip install xgboost

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/72.0 MB 1.3 MB/s eta 0:00:56
    --------------------------------------- 1.0/72.0 MB 1.2 MB/s eta 0:00:58
    --------------------------------------- 1.3/72.0 MB 1.2 MB/s eta 0:01:02
    --------------------------------------- 1.6/72.0 MB 1.3 MB/s eta 0:00:55
   - -------------------------------------- 1.8/72.0 MB 1.3 MB/s eta 0:00:55
   - -------------------------------------- 2.1/72.0 MB 1.3 MB/s eta 0:00:54
   - -------------------------------------- 2.4/72.0 MB 1.3 MB/s eta 0:00:54
   - -------------------------------------- 2.6/72.0 MB 1.3 MB/s eta 0:00:54
   - -------------------------------------- 2.9/72.0 MB 1.3 MB/s eta 0:00:54
   - -------------------------------------- 3.1/72.0 MB 1.3 MB/s eta 0:00:53
   - ---------------

In [2]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pickle


df = pd.read_csv('cleaned_dataset.csv')

# Display the first few rows
df.head()

,ID,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,...,"Amenities_Pool, Playground, Gym, Clubhouse, Garden","Amenities_Pool, Playground, Gym, Garden","Amenities_Pool, Playground, Gym, Garden, Clubhouse",Facing_North,Facing_South,Facing_West,Owner_Type_Builder,Owner_Type_Owner,Availability_Status_Under_Construction,Future_Price_in_Lakhs
0,-1.731995,-1.412257,1.494463,1.766636,0.020748,-1.562275,0.785028,-1.671033,1.562275,1.563892,...,False,False,False,False,False,True,False,True,False,1.766636
1,-1.731981,0.001008,-0.451913,-0.340268,0.014845,-0.002239,0.673285,0.519590,0.002239,0.869038,...,False,False,False,True,False,False,True,False,True,-0.340268
2,-1.731967,-0.705625,0.595002,-0.424261,-0.005318,-0.002239,0.449799,1.326661,0.002239,1.216465,...,False,False,False,False,True,False,False,False,False,-0.424261
3,-1.731954,-0.705625,-0.143081,0.409937,-0.034917,-1.562275,0.673285,1.211365,1.562275,-0.173242,...,False,False,False,True,False,False,True,False,False,0.409937
4,-1.731940,0.707640,1.562456,-0.430634,0.000698,-0.002239,-1.338093,-1.555737,0.002239,-0.520669,...,False,False,False,False,False,False,True,False,False,-0.430634


## Data Preparation

For regression, we assume 'Price_in_Lakhs' as the current price and create a 'Future_Price' by adding 5% annual growth (compounded) for 5 years.

For classification, we create a binary target 'Good_Investment' where 1 if Price_per_SqFt > median, else 0.

In [3]:
# Prepare data for regression
# Assume future price = current price * (1.05)^5
df['Future_Price'] = df['Price_in_Lakhs'] * (1.05 ** 5)

# Features for regression (exclude price-related columns)
features_reg = [col for col in df.columns if col not in ['Price_in_Lakhs', 'Price_per_SqFt', 'Future_Price']]
X_reg = df[features_reg]
y_reg = df['Future_Price']

# Prepare data for classification
# Good investment if Price_per_SqFt > median
median_ppsf = df['Price_per_SqFt'].median()
df['Good_Investment'] = (df['Price_per_SqFt'] > median_ppsf).astype(int)

# Features for classification (exclude price-related columns)
features_clf = [col for col in df.columns if col not in ['Price_in_Lakhs', 'Price_per_SqFt', 'Future_Price', 'Good_Investment']]
X_clf = df[features_clf]
y_clf = df['Good_Investment']

# Train-test split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

print('Regression shapes:', X_train_reg.shape, X_test_reg.shape)
print('Classification shapes:', X_train_clf.shape, X_test_clf.shape)

Regression shapes: (183984, 907) (45996, 907)
Classification shapes: (183984, 907) (45996, 907)


## Regression Models

### Linear Regression
- **Why required**: Simple baseline model to understand linear relationships.
- **How it works**: Fits a straight line to minimize the sum of squared errors between predicted and actual values.


In [4]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train_reg, y_train_reg)
y_pred_lr = lr.predict(X_test_reg)

# Evaluation
rmse_lr = np.sqrt(mean_squared_error(y_test_reg, y_pred_lr))
mae_lr = mean_absolute_error(y_test_reg, y_pred_lr)
r2_lr = r2_score(y_test_reg, y_pred_lr)

print('Linear Regression - RMSE:', rmse_lr, 'MAE:', mae_lr, 'R²:', r2_lr)

Linear Regression - RMSE: 2.4764997429743748e-15 MAE: 2.054702557973967e-15 R²: 1.0


### Random Forest Regressor
- **Why required**: Handles non-linear relationships and feature interactions well.
- **How it works**: Builds multiple decision trees and averages their predictions to reduce overfitting.

In [5]:
# Random Forest Regressor
rf_reg = RandomForestRegressor(random_state=42)
rf_reg.fit(X_train_reg, y_train_reg)
y_pred_rf_reg = rf_reg.predict(X_test_reg)

# Evaluation
rmse_rf_reg = np.sqrt(mean_squared_error(y_test_reg, y_pred_rf_reg))
mae_rf_reg = mean_absolute_error(y_test_reg, y_pred_rf_reg)
r2_rf_reg = r2_score(y_test_reg, y_pred_rf_reg)

print('Random Forest Regressor - RMSE:', rmse_rf_reg, 'MAE:', mae_rf_reg, 'R²:', r2_rf_reg)

Random Forest Regressor - RMSE: 2.2182343470986457e-05 MAE: 1.3679265274077885e-05 R²: 0.999999999698231


### XGBoost Regressor
- **Why required**: Advanced model for high performance, handles missing values and regularization.
- **How it works**: Uses gradient boosting to build trees sequentially, correcting errors of previous trees.

In [6]:
# XGBoost Regressor
xgb_reg = XGBRegressor(random_state=42)
xgb_reg.fit(X_train_reg, y_train_reg)
y_pred_xgb_reg = xgb_reg.predict(X_test_reg)

# Evaluation
rmse_xgb_reg = np.sqrt(mean_squared_error(y_test_reg, y_pred_xgb_reg))
mae_xgb_reg = mean_absolute_error(y_test_reg, y_pred_xgb_reg)
r2_xgb_reg = r2_score(y_test_reg, y_pred_xgb_reg)

print('XGBoost Regressor - RMSE:', rmse_xgb_reg, 'MAE:', mae_xgb_reg, 'R²:', r2_xgb_reg)

XGBoost Regressor - RMSE: 0.005132967555861375 MAE: 0.0044035272127941755 R²: 0.9999838416519061


## Classification Models

### Logistic Regression
- **Why required**: Simple baseline for binary classification.
- **How it works**: Uses a sigmoid function to predict probabilities, classifying based on a threshold.

In [7]:
# Logistic Regression
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_clf, y_train_clf)
y_pred_log = log_reg.predict(X_test_clf)

# Evaluation
acc_log = accuracy_score(y_test_clf, y_pred_log)
prec_log = precision_score(y_test_clf, y_pred_log)
rec_log = recall_score(y_test_clf, y_pred_log)
f1_log = f1_score(y_test_clf, y_pred_log)
cm_log = confusion_matrix(y_test_clf, y_pred_log)

print('Logistic Regression - Accuracy:', acc_log, 'Precision:', prec_log, 'Recall:', rec_log, 'F1:', f1_log)
print('Confusion Matrix:')
print(cm_log)

Logistic Regression - Accuracy: 0.5225671797547613 Precision: 0.5232573122020359 Recall: 0.5287071995833876 F1: 0.5259681388421189
Confusion Matrix:
[[11853 11100]
 [10860 12183]]


### Random Forest Classifier
- **Why required**: Handles complex patterns and reduces overfitting.
- **How it works**: Builds multiple decision trees and uses majority voting for classification.

In [8]:
# Random Forest Classifier
rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train_clf, y_train_clf)
y_pred_rf_clf = rf_clf.predict(X_test_clf)

# Evaluation
acc_rf_clf = accuracy_score(y_test_clf, y_pred_rf_clf)
prec_rf_clf = precision_score(y_test_clf, y_pred_rf_clf)
rec_rf_clf = recall_score(y_test_clf, y_pred_rf_clf)
f1_rf_clf = f1_score(y_test_clf, y_pred_rf_clf)
cm_rf_clf = confusion_matrix(y_test_clf, y_pred_rf_clf)

print('Random Forest Classifier - Accuracy:', acc_rf_clf, 'Precision:', prec_rf_clf, 'Recall:', rec_rf_clf, 'F1:', f1_rf_clf)
print('Confusion Matrix:')
print(cm_rf_clf)

Random Forest Classifier - Accuracy: 0.9926297938951213 Precision: 0.9963273871983211 Recall: 0.9889337325869028 F1: 0.9926167918980725
Confusion Matrix:
[[22869    84]
 [  255 22788]]


### XGBoost Classifier
- **Why required**: High-performance model for classification tasks.
- **How it works**: Uses gradient boosting to build trees that minimize classification errors.

In [9]:
# XGBoost Classifier
xgb_clf = XGBClassifier(random_state=42)
xgb_clf.fit(X_train_clf, y_train_clf)
y_pred_xgb_clf = xgb_clf.predict(X_test_clf)

# Evaluation
acc_xgb_clf = accuracy_score(y_test_clf, y_pred_xgb_clf)
prec_xgb_clf = precision_score(y_test_clf, y_pred_xgb_clf)
rec_xgb_clf = recall_score(y_test_clf, y_pred_xgb_clf)
f1_xgb_clf = f1_score(y_test_clf, y_pred_xgb_clf)
cm_xgb_clf = confusion_matrix(y_test_clf, y_pred_xgb_clf)

print('XGBoost Classifier - Accuracy:', acc_xgb_clf, 'Precision:', prec_xgb_clf, 'Recall:', rec_xgb_clf, 'F1:', f1_xgb_clf)
print('Confusion Matrix:')
print(cm_xgb_clf)

XGBoost Classifier - Accuracy: 0.9979563440299156 Precision: 0.9976148141723405 Recall: 0.9983075120427027 F1: 0.9979610429048631
Confusion Matrix:
[[22898    55]
 [   39 23004]]


## Comparison Tables

### Regression Comparison

In [10]:
# Regression Comparison Table
reg_results = {
    'Model': ['Linear Regression', 'Random Forest Regressor', 'XGBoost Regressor'],
    'RMSE': [rmse_lr, rmse_rf_reg, rmse_xgb_reg],
    'MAE': [mae_lr, mae_rf_reg, mae_xgb_reg],
    'R²': [r2_lr, r2_rf_reg, r2_xgb_reg]
}
reg_df = pd.DataFrame(reg_results)
print(reg_df)

                     Model          RMSE           MAE        R²
0        Linear Regression  2.476500e-15  2.054703e-15  1.000000
1  Random Forest Regressor  2.218234e-05  1.367927e-05  1.000000
2        XGBoost Regressor  5.132968e-03  4.403527e-03  0.999984


### Classification Comparison

In [11]:
# Classification Comparison Table
clf_results = {
    'Model': ['Logistic Regression', 'Random Forest Classifier', 'XGBoost Classifier'],
    'Accuracy': [acc_log, acc_rf_clf, acc_xgb_clf],
    'Precision': [prec_log, prec_rf_clf, prec_xgb_clf],
    'Recall': [rec_log, rec_rf_clf, rec_xgb_clf],
    'F1-Score': [f1_log, f1_rf_clf, f1_xgb_clf]
}
clf_df = pd.DataFrame(clf_results)
print(clf_df)

                      Model  Accuracy  Precision    Recall  F1-Score
0       Logistic Regression  0.522567   0.523257  0.528707  0.525968
1  Random Forest Classifier  0.992630   0.996327  0.988934  0.992617
2        XGBoost Classifier  0.997956   0.997615  0.998308  0.997961


In [12]:
import joblib

# Save Regression Models
joblib.dump(lr, "linear_model.pkl")
joblib.dump(rf_reg, "random_forest_reg.pkl")
joblib.dump(xgb_reg, "xgboost_reg.pkl")

# Save Classification Models
joblib.dump(log_reg, "logistic_model.pkl")
joblib.dump(rf_clf, "random_forest_clf.pkl")
joblib.dump(xgb_clf, "xgboost_clf.pkl")

print("All models saved successfully!")


All models saved successfully!


In [13]:
print(X_test_reg.columns.tolist())


['ID', 'BHK', 'Size_in_SqFt', 'Year_Built', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'State_Assam', 'State_Bihar', 'State_Chhattisgarh', 'State_Delhi', 'State_Gujarat', 'State_Haryana', 'State_Jharkhand', 'State_Karnataka', 'State_Kerala', 'State_Madhya Pradesh', 'State_Maharashtra', 'State_Odisha', 'State_Punjab', 'State_Rajasthan', 'State_Tamil Nadu', 'State_Telangana', 'State_Uttar Pradesh', 'State_Uttarakhand', 'State_West Bengal', 'City_Amritsar', 'City_Bangalore', 'City_Bhopal', 'City_Bhubaneswar', 'City_Bilaspur', 'City_Chennai', 'City_Coimbatore', 'City_Cuttack', 'City_Dehradun', 'City_Durgapur', 'City_Dwarka', 'City_Faridabad', 'City_Gaya', 'City_Gurgaon', 'City_Guwahati', 'City_Haridwar', 'City_Hyderabad', 'City_Indore', 'City_Jaipur', 'City_Jamshedpur', 'City_Jodhpur', 'City_Kochi', 'City_Kolkata', 'City_Lucknow', 'City_Ludhiana', 'City_Mangalore', 'City_Mumbai', 'City_Mysore', 'City_Nagpur', 'City_New Delhi', 'City_Noida', 'City_P